# Evaluation Metrics for Summarization Models (Original and V2)

This notebook calculates intrinsic evaluation metrics for extractive and abstractive summarization:

1. **Key-Entity Overlap (KE Overlap)**: Measures overlap of important entities between summary and source
2. **Compression Ratio**: Ratio of summary length to original source length
3. **Redundancy Check**: Measures repetition and redundancy within the summary

## Input:
- Original versions:
  - `OP_SUMMARIES/textrank_overall_summary.txt`
  - `OP_SUMMARIES/lexrank_overall_summary.txt`
  - `OP_SUMMARIES/distilbart_overall_summary.txt`
  - `OP_SUMMARIES/pegasus_overall_summary.txt`
- V2 versions (with modified parameters):
  - `OP_SUMMARIES/textrank_v2_overall_summary.txt`
  - `OP_SUMMARIES/lexrank_v2_overall_summary.txt`
  - `OP_SUMMARIES/distilbart_v2_overall_summary.txt`
  - `OP_SUMMARIES/pegasus_v2_overall_summary.txt`
- Source documents: All `no_footer_*.txt` files from `pre_proc_op/`

## Note:
These are **intrinsic metrics** - they don't require ground truth/reference summaries.


In [14]:
# Import required libraries
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# For entity extraction and text processing
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tag import pos_tag
from nltk.chunk import ne_chunk
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger', quiet=True)

try:
    nltk.data.find('chunkers/maxent_ne_chunker')
except LookupError:
    nltk.download('maxent_ne_chunker', quiet=True)

try:
    nltk.data.find('corpora/words')
except LookupError:
    nltk.download('words', quiet=True)

print("Libraries imported successfully")


Libraries imported successfully


In [15]:
# Configuration - resolve project root (notebook lives in eval/ subfolder)
if (Path.cwd() / "OP_SUMMARIES").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "OP_SUMMARIES").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()
EVAL_OUTPUT_DIR = PROJECT_ROOT / "eval"

op_summaries_path = PROJECT_ROOT / "OP_SUMMARIES"

# Original versions
textrank_summary_path = op_summaries_path / "textrank_overall_summary.txt"
lexrank_summary_path = op_summaries_path / "lexrank_overall_summary.txt"
distilbart_summary_path = op_summaries_path / "distilbart_overall_summary.txt"
pegasus_summary_path = op_summaries_path / "pegasus_overall_summary.txt"

# V2 versions (with modified parameters)
textrank_v2_summary_path = op_summaries_path / "textrank_v2_overall_summary.txt"
lexrank_v2_summary_path = op_summaries_path / "lexrank_v2_overall_summary.txt"
distilbart_v2_summary_path = op_summaries_path / "distilbart_v2_overall_summary.txt"
pegasus_v2_summary_path = op_summaries_path / "pegasus_v2_overall_summary.txt"

source_base_path = PROJECT_ROOT / "pre_proc_op"
years = ["2020", "2021", "2022", "2023", "2024"]

print("Configuration:")
print(f"OP Summaries folder: {op_summaries_path}")
print(f"\nOriginal versions:")
print(f"  TextRank: {textrank_summary_path}")
print(f"  LexRank: {lexrank_summary_path}")
print(f"  DistilBART: {distilbart_summary_path}")
print(f"  Pegasus: {pegasus_summary_path}")
print(f"\nV2 versions:")
print(f"  TextRank V2: {textrank_v2_summary_path}")
print(f"  LexRank V2: {lexrank_v2_summary_path}")
print(f"  DistilBART V2: {distilbart_v2_summary_path}")
print(f"  Pegasus V2: {pegasus_v2_summary_path}")
print(f"\nSource documents: {source_base_path}")


Configuration:
OP Summaries folder: OP_SUMMARIES

Original versions:
  TextRank: OP_SUMMARIES\textrank_overall_summary.txt
  LexRank: OP_SUMMARIES\lexrank_overall_summary.txt
  DistilBART: OP_SUMMARIES\distilbart_overall_summary.txt
  Pegasus: OP_SUMMARIES\pegasus_overall_summary.txt

V2 versions:
  TextRank V2: OP_SUMMARIES\textrank_v2_overall_summary.txt
  LexRank V2: OP_SUMMARIES\lexrank_v2_overall_summary.txt
  DistilBART V2: OP_SUMMARIES\distilbart_v2_overall_summary.txt
  Pegasus V2: OP_SUMMARIES\pegasus_v2_overall_summary.txt

Source documents: pre_proc_op


In [16]:
# Helper function to load summary text (OP_SUMMARIES files are already cleaned, no headers)
def load_summary(file_path):
    """
    Load summary text from file (OP_SUMMARIES files are already cleaned, no headers)
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            return content
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return ""

# Load original summaries
textrank_summary = load_summary(textrank_summary_path)
lexrank_summary = load_summary(lexrank_summary_path)
distilbart_summary = load_summary(distilbart_summary_path)
pegasus_summary = load_summary(pegasus_summary_path)

# Load V2 summaries
textrank_v2_summary = load_summary(textrank_v2_summary_path)
lexrank_v2_summary = load_summary(lexrank_v2_summary_path)
distilbart_v2_summary = load_summary(distilbart_v2_summary_path)
pegasus_v2_summary = load_summary(pegasus_v2_summary_path)

print("Original summaries loaded:")
print(f"  TextRank: {len(textrank_summary)} characters")
print(f"  LexRank: {len(lexrank_summary)} characters")
print(f"  DistilBART: {len(distilbart_summary)} characters")
print(f"  Pegasus: {len(pegasus_summary)} characters")
print(f"\nV2 summaries loaded:")
print(f"  TextRank V2: {len(textrank_v2_summary)} characters")
print(f"  LexRank V2: {len(lexrank_v2_summary)} characters")
print(f"  DistilBART V2: {len(distilbart_v2_summary)} characters")
print(f"  Pegasus V2: {len(pegasus_v2_summary)} characters")


Original summaries loaded:
  TextRank: 751 characters
  LexRank: 485 characters
  DistilBART: 977 characters
  Pegasus: 1718 characters

V2 summaries loaded:
  TextRank V2: 1697 characters
  LexRank V2: 1693 characters
  DistilBART V2: 2456 characters
  Pegasus V2: 879 characters


In [17]:
# Load all source documents (no_footer_ files)
def load_source_documents():
    """
    Load all no_footer_*.txt files from all years
    """
    all_text = ""
    total_files = 0
    
    for year in years:
        year_path = source_base_path / year
        if not year_path.exists():
            continue
        
        no_footer_files = list(year_path.glob("no_footer_*.txt"))
        
        for file_path in no_footer_files:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read().strip()
                    if content:
                        all_text += content + " "
                        total_files += 1
            except Exception as e:
                print(f"Error reading {file_path.name}: {e}")
    
    return all_text.strip(), total_files

source_text, num_source_files = load_source_documents()
source_length = len(source_text)

print(f"Source documents loaded:")
print(f"  Total files: {num_source_files}")
print(f"  Total characters: {source_length:,}")


Source documents loaded:
  Total files: 82
  Total characters: 76,488


## 1. Compression Ratio

Compression ratio = Summary Length / Source Length

Lower ratio = more compressed (better compression)


In [18]:
# Calculate Compression Ratio
def calculate_compression_ratio(summary_text, source_text):
    """
    Calculate compression ratio: summary_length / source_length
    Also calculate compression percentage: (1 - ratio) * 100
    """
    summary_len = len(summary_text)
    source_len = len(source_text)
    
    if source_len == 0:
        return None, None
    
    ratio = summary_len / source_len
    compression_percent = (1 - ratio) * 100
    
    return ratio, compression_percent

# Calculate for all summaries (original and v2)
textrank_ratio, textrank_compression = calculate_compression_ratio(textrank_summary, source_text)
lexrank_ratio, lexrank_compression = calculate_compression_ratio(lexrank_summary, source_text)
distilbart_ratio, distilbart_compression = calculate_compression_ratio(distilbart_summary, source_text)
pegasus_ratio, pegasus_compression = calculate_compression_ratio(pegasus_summary, source_text)

textrank_v2_ratio, textrank_v2_compression = calculate_compression_ratio(textrank_v2_summary, source_text)
lexrank_v2_ratio, lexrank_v2_compression = calculate_compression_ratio(lexrank_v2_summary, source_text)
distilbart_v2_ratio, distilbart_v2_compression = calculate_compression_ratio(distilbart_v2_summary, source_text)
pegasus_v2_ratio, pegasus_v2_compression = calculate_compression_ratio(pegasus_v2_summary, source_text)

print("COMPRESSION RATIO:")
print("=" * 70)
print("ORIGINAL VERSIONS:")
print("-" * 70)
print(f"TextRank:")
print(f"  Summary length: {len(textrank_summary):,} characters")
print(f"  Source length: {source_length:,} characters")
print(f"  Compression ratio: {textrank_ratio:.4f}")
print(f"  Compression: {textrank_compression:.2f}%")
print(f"\nLexRank:")
print(f"  Summary length: {len(lexrank_summary):,} characters")
print(f"  Source length: {source_length:,} characters")
print(f"  Compression ratio: {lexrank_ratio:.4f}")
print(f"  Compression: {lexrank_compression:.2f}%")
print(f"\nDistilBART:")
print(f"  Summary length: {len(distilbart_summary):,} characters")
print(f"  Source length: {source_length:,} characters")
print(f"  Compression ratio: {distilbart_ratio:.4f}")
print(f"  Compression: {distilbart_compression:.2f}%")
print(f"\nPegasus:")
print(f"  Summary length: {len(pegasus_summary):,} characters")
print(f"  Source length: {source_length:,} characters")
print(f"  Compression ratio: {pegasus_ratio:.4f}")
print(f"  Compression: {pegasus_compression:.2f}%")

print(f"\nV2 VERSIONS:")
print("-" * 70)
print(f"TextRank V2:")
print(f"  Summary length: {len(textrank_v2_summary):,} characters")
print(f"  Source length: {source_length:,} characters")
print(f"  Compression ratio: {textrank_v2_ratio:.4f}")
print(f"  Compression: {textrank_v2_compression:.2f}%")
print(f"\nLexRank V2:")
print(f"  Summary length: {len(lexrank_v2_summary):,} characters")
print(f"  Source length: {source_length:,} characters")
print(f"  Compression ratio: {lexrank_v2_ratio:.4f}")
print(f"  Compression: {lexrank_v2_compression:.2f}%")
print(f"\nDistilBART V2:")
print(f"  Summary length: {len(distilbart_v2_summary):,} characters")
print(f"  Source length: {source_length:,} characters")
print(f"  Compression ratio: {distilbart_v2_ratio:.4f}")
print(f"  Compression: {distilbart_v2_compression:.2f}%")
print(f"\nPegasus V2:")
print(f"  Summary length: {len(pegasus_v2_summary):,} characters")
print(f"  Source length: {source_length:,} characters")
print(f"  Compression ratio: {pegasus_v2_ratio:.4f}")
print(f"  Compression: {pegasus_v2_compression:.2f}%")


COMPRESSION RATIO:
ORIGINAL VERSIONS:
----------------------------------------------------------------------
TextRank:
  Summary length: 751 characters
  Source length: 76,488 characters
  Compression ratio: 0.0098
  Compression: 99.02%

LexRank:
  Summary length: 485 characters
  Source length: 76,488 characters
  Compression ratio: 0.0063
  Compression: 99.37%

DistilBART:
  Summary length: 977 characters
  Source length: 76,488 characters
  Compression ratio: 0.0128
  Compression: 98.72%

Pegasus:
  Summary length: 1,718 characters
  Source length: 76,488 characters
  Compression ratio: 0.0225
  Compression: 97.75%

V2 VERSIONS:
----------------------------------------------------------------------
TextRank V2:
  Summary length: 1,697 characters
  Source length: 76,488 characters
  Compression ratio: 0.0222
  Compression: 97.78%

LexRank V2:
  Summary length: 1,693 characters
  Source length: 76,488 characters
  Compression ratio: 0.0221
  Compression: 97.79%

DistilBART V2:
  Summa

## 2. Key-Entity Overlap (KE Overlap)

Measures the overlap of important entities (named entities, key terms) between summary and source.

KE Overlap = (Common Entities / Total Unique Entities in Summary) * 100


In [19]:
# Extract key entities from text
def extract_entities(text):
    """
    Extract key entities from text:
    1. Named entities (using NLTK)
    2. Important terms (capitalized words, car numbers, driver names, etc.)
    3. Key technical terms
    """
    entities = set()
    
    # Tokenize and tag
    try:
        tokens = word_tokenize(text)
        tagged = pos_tag(tokens)
        
        # Extract named entities
        tree = ne_chunk(tagged, binary=False)
        for subtree in tree:
            if isinstance(subtree, nltk.Tree):
                entity = ' '.join([word for word, tag in subtree.leaves()])
                entities.add(entity.lower())
    except Exception as e:
        pass  # Silently continue if NER fails
    
    # Extract car numbers (Car 44, Car 63, etc.)
    car_numbers = re.findall(r'car\s+\d+', text, re.IGNORECASE)
    for car in car_numbers:
        entities.add(car.lower())
    
    # Extract driver names (common patterns)
    driver_patterns = [
        r'lewis\s+hamilton',
        r'george\s+russell',
        r'valtteri\s+bottas',
        r'hamilton',
        r'russell',
        r'bottas'
    ]
    for pattern in driver_patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            entities.add(match.lower())
    
    # Extract important capitalized terms (likely entities)
    capitalized_terms = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text)
    for term in capitalized_terms:
        # Filter out common words
        if len(term) > 3 and term.lower() not in ['the', 'and', 'for', 'are', 'was', 'were']:
            entities.add(term.lower())
    
    # Extract technical terms (FIA, Article, etc.)
    technical_terms = re.findall(r'\b(?:fia|article|art\.|regulation|steward|penalty|fine|decision|offence|infringement)\b', text, re.IGNORECASE)
    for term in technical_terms:
        entities.add(term.lower())
    
    return entities

# Calculate Key-Entity Overlap
def calculate_ke_overlap(summary_text, source_text):
    """
    Calculate Key-Entity Overlap between summary and source
    """
    summary_entities = extract_entities(summary_text)
    source_entities = extract_entities(source_text)
    
    if len(summary_entities) == 0:
        return 0.0, summary_entities, source_entities, set()
    
    # Find common entities
    common_entities = summary_entities.intersection(source_entities)
    
    # KE Overlap = (Common Entities / Total Entities in Summary) * 100
    overlap_score = (len(common_entities) / len(summary_entities)) * 100
    
    return overlap_score, summary_entities, source_entities, common_entities

# Calculate for all summaries (original and v2)
textrank_ke, textrank_sum_ents, textrank_src_ents, textrank_common = calculate_ke_overlap(textrank_summary, source_text)
lexrank_ke, lexrank_sum_ents, lexrank_src_ents, lexrank_common = calculate_ke_overlap(lexrank_summary, source_text)
distilbart_ke, distilbart_sum_ents, distilbart_src_ents, distilbart_common = calculate_ke_overlap(distilbart_summary, source_text)
pegasus_ke, pegasus_sum_ents, pegasus_src_ents, pegasus_common = calculate_ke_overlap(pegasus_summary, source_text)

textrank_v2_ke, textrank_v2_sum_ents, textrank_v2_src_ents, textrank_v2_common = calculate_ke_overlap(textrank_v2_summary, source_text)
lexrank_v2_ke, lexrank_v2_sum_ents, lexrank_v2_src_ents, lexrank_v2_common = calculate_ke_overlap(lexrank_v2_summary, source_text)
distilbart_v2_ke, distilbart_v2_sum_ents, distilbart_v2_src_ents, distilbart_v2_common = calculate_ke_overlap(distilbart_v2_summary, source_text)
pegasus_v2_ke, pegasus_v2_sum_ents, pegasus_v2_src_ents, pegasus_v2_common = calculate_ke_overlap(pegasus_v2_summary, source_text)
print("KEY-ENTITY OVERLAP:")
print("=" * 70)
print("ORIGINAL VERSIONS:")
print("-" * 70)
print(f"TextRank:")
print(f"  Entities in summary: {len(textrank_sum_ents)}")
print(f"  Entities in source: {len(textrank_src_ents)}")
print(f"  Common entities: {len(textrank_common)}")
print(f"  KE Overlap: {textrank_ke:.2f}%")
print(f"\nLexRank:")
print(f"  Entities in summary: {len(lexrank_sum_ents)}")
print(f"  Entities in source: {len(lexrank_src_ents)}")
print(f"  Common entities: {len(lexrank_common)}")
print(f"  KE Overlap: {lexrank_ke:.2f}%")
print(f"\nDistilBART:")
print(f"  Entities in summary: {len(distilbart_sum_ents)}")
print(f"  Entities in source: {len(distilbart_src_ents)}")
print(f"  Common entities: {len(distilbart_common)}")
print(f"  KE Overlap: {distilbart_ke:.2f}%")
print(f"\nPegasus:")
print(f"  Entities in summary: {len(pegasus_sum_ents)}")
print(f"  Entities in source: {len(pegasus_src_ents)}")
print(f"  Common entities: {len(pegasus_common)}")
print(f"  KE Overlap: {pegasus_ke:.2f}%")

print(f"\nV2 VERSIONS:")
print("-" * 70)
print(f"TextRank V2:")
print(f"  Entities in summary: {len(textrank_v2_sum_ents)}")
print(f"  Entities in source: {len(textrank_v2_src_ents)}")
print(f"  Common entities: {len(textrank_v2_common)}")
print(f"  KE Overlap: {textrank_v2_ke:.2f}%")
print(f"\nLexRank V2:")
print(f"  Entities in summary: {len(lexrank_v2_sum_ents)}")
print(f"  Entities in source: {len(lexrank_v2_src_ents)}")
print(f"  Common entities: {len(lexrank_v2_common)}")
print(f"  KE Overlap: {lexrank_v2_ke:.2f}%")
print(f"\nDistilBART V2:")
print(f"  Entities in summary: {len(distilbart_v2_sum_ents)}")
print(f"  Entities in source: {len(distilbart_v2_src_ents)}")
print(f"  Common entities: {len(distilbart_v2_common)}")
print(f"  KE Overlap: {distilbart_v2_ke:.2f}%")
print(f"\nPegasus V2:")
print(f"  Entities in summary: {len(pegasus_v2_sum_ents)}")
print(f"  Entities in source: {len(pegasus_v2_src_ents)}")
print(f"  Common entities: {len(pegasus_v2_common)}")
print(f"  KE Overlap: {distilbart_v2_ke:.2f}%")

# Show sample common entities
print(f"\nSample common entities (TextRank): {list(textrank_common)[:10]}")
print(f"Sample common entities (LexRank): {list(lexrank_common)[:10]}")
print(f"Sample common entities (DistilBART): {list(distilbart_common)[:10]}")
print(f"Sample common entities (Pegasus): {list(pegasus_common)[:10]}")
print(f"Sample common entities (TextRank V2): {list(textrank_v2_common)[:10]}")
print(f"Sample common entities (LexRank V2): {list(lexrank_v2_common)[:10]}")
print(f"Sample common entities (DistilBART V2): {list(distilbart_v2_common)[:10]}")
print(f"Sample common entities (Pegasus V2): {list(pegasus_v2_common)[:10]}")

KEY-ENTITY OVERLAP:
ORIGINAL VERSIONS:
----------------------------------------------------------------------
TextRank:
  Entities in summary: 9
  Entities in source: 387
  Common entities: 9
  KE Overlap: 100.00%

LexRank:
  Entities in summary: 19
  Entities in source: 387
  Common entities: 19
  KE Overlap: 100.00%

DistilBART:
  Entities in summary: 19
  Entities in source: 387
  Common entities: 17
  KE Overlap: 89.47%

Pegasus:
  Entities in summary: 33
  Entities in source: 387
  Common entities: 27
  KE Overlap: 81.82%

V2 VERSIONS:
----------------------------------------------------------------------
TextRank V2:
  Entities in summary: 23
  Entities in source: 387
  Common entities: 16
  KE Overlap: 69.57%

LexRank V2:
  Entities in summary: 35
  Entities in source: 387
  Common entities: 28
  KE Overlap: 80.00%

DistilBART V2:
  Entities in summary: 42
  Entities in source: 387
  Common entities: 30
  KE Overlap: 71.43%

Pegasus V2:
  Entities in summary: 21
  Entities in so

## 3. Redundancy Check

Measures repetition and redundancy within the summary using:
- Sentence similarity (cosine similarity between sentences)
- N-gram repetition
- Average sentence similarity


In [20]:
# Calculate Redundancy
def calculate_redundancy(summary_text):
    """
    Calculate redundancy metrics:
    1. Average sentence similarity (higher = more redundant)
    2. N-gram repetition rate
    3. Redundancy score (0-100, higher = more redundant)
    """
    if not summary_text or len(summary_text.strip()) < 10:
        return None, None, None
    
    # Split into sentences
    sentences = sent_tokenize(summary_text)
    
    if len(sentences) < 2:
        return 0.0, 0.0, 0.0  # No redundancy if only one sentence
    
    # Calculate sentence similarity using TF-IDF
    vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
    try:
        sentence_vectors = vectorizer.fit_transform(sentences)
        similarity_matrix = cosine_similarity(sentence_vectors)
        
        # Average similarity (excluding diagonal)
        np.fill_diagonal(similarity_matrix, 0)  # Remove self-similarity
        num_pairs = len(sentences) * (len(sentences) - 1) / 2
        avg_similarity = similarity_matrix.sum() / (2 * num_pairs) if num_pairs > 0 else 0
    except Exception as e:
        print(f"Error calculating similarity: {e}")
        avg_similarity = 0.0
    
    # Calculate n-gram repetition (bigrams and trigrams)
    words = word_tokenize(summary_text.lower())
    
    # Bigram repetition
    bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
    bigram_counts = Counter(bigrams)
    repeated_bigrams = sum(1 for count in bigram_counts.values() if count > 1)
    bigram_repetition = (repeated_bigrams / len(bigram_counts)) * 100 if len(bigram_counts) > 0 else 0
    
    # Trigram repetition
    trigrams = [f"{words[i]} {words[i+1]} {words[i+2]}" for i in range(len(words)-2)]
    trigram_counts = Counter(trigrams)
    repeated_trigrams = sum(1 for count in trigram_counts.values() if count > 1)
    trigram_repetition = (repeated_trigrams / len(trigram_counts)) * 100 if len(trigram_counts) > 0 else 0
    
    # Overall redundancy score (weighted combination)
    # Higher score = more redundant
    redundancy_score = (avg_similarity * 50) + (bigram_repetition * 0.3) + (trigram_repetition * 0.2)
    redundancy_score = min(100, redundancy_score)  # Cap at 100
    
    return avg_similarity, bigram_repetition, redundancy_score

# Calculate for all summaries (original and v2)
textrank_avg_sim, textrank_bigram, textrank_redundancy = calculate_redundancy(textrank_summary)
lexrank_avg_sim, lexrank_bigram, lexrank_redundancy = calculate_redundancy(lexrank_summary)
distilbart_avg_sim, distilbart_bigram, distilbart_redundancy = calculate_redundancy(distilbart_summary)
pegasus_avg_sim, pegasus_bigram, pegasus_redundancy = calculate_redundancy(pegasus_summary)

textrank_v2_avg_sim, textrank_v2_bigram, textrank_v2_redundancy = calculate_redundancy(textrank_v2_summary)
lexrank_v2_avg_sim, lexrank_v2_bigram, lexrank_v2_redundancy = calculate_redundancy(lexrank_v2_summary)
distilbart_v2_avg_sim, distilbart_v2_bigram, distilbart_v2_redundancy = calculate_redundancy(distilbart_v2_summary)
pegasus_v2_avg_sim, pegasus_v2_bigram, pegasus_v2_redundancy = calculate_redundancy(pegasus_v2_summary)

print("REDUNDANCY CHECK:")
print("=" * 70)
print("ORIGINAL VERSIONS:")
print("-" * 70)
print(f"TextRank:")
print(f"  Average sentence similarity: {textrank_avg_sim:.4f}")
print(f"  Bigram repetition rate: {textrank_bigram:.2f}%")
print(f"  Redundancy score: {textrank_redundancy:.2f} (higher = more redundant)")
print(f"\nLexRank:")
print(f"  Average sentence similarity: {lexrank_avg_sim:.4f}")
print(f"  Bigram repetition rate: {lexrank_bigram:.2f}%")
print(f"  Redundancy score: {lexrank_redundancy:.2f} (higher = more redundant)")
print(f"\nDistilBART:")
print(f"  Average sentence similarity: {distilbart_avg_sim:.4f}")
print(f"  Bigram repetition rate: {distilbart_bigram:.2f}%")
print(f"  Redundancy score: {distilbart_redundancy:.2f} (higher = more redundant)")
print(f"\nPegasus:")
print(f"  Average sentence similarity: {pegasus_avg_sim:.4f}")
print(f"  Bigram repetition rate: {pegasus_bigram:.2f}%")
print(f"  Redundancy score: {pegasus_redundancy:.2f} (higher = more redundant)")

print(f"\nV2 VERSIONS:")
print("-" * 70)
print(f"TextRank V2:")
print(f"  Average sentence similarity: {textrank_v2_avg_sim:.4f}")
print(f"  Bigram repetition rate: {textrank_v2_bigram:.2f}%")
print(f"  Redundancy score: {textrank_v2_redundancy:.2f} (higher = more redundant)")
print(f"\nLexRank V2:")
print(f"  Average sentence similarity: {lexrank_v2_avg_sim:.4f}")
print(f"  Bigram repetition rate: {lexrank_v2_bigram:.2f}%")
print(f"  Redundancy score: {lexrank_v2_redundancy:.2f} (higher = more redundant)")
print(f"\nDistilBART V2:")
print(f"  Average sentence similarity: {distilbart_v2_avg_sim:.4f}")
print(f"  Bigram repetition rate: {distilbart_v2_bigram:.2f}%")
print(f"  Redundancy score: {distilbart_v2_redundancy:.2f} (higher = more redundant)")
print(f"\nPegasus V2:")
print(f"  Average sentence similarity: {pegasus_v2_avg_sim:.4f}")
print(f"  Bigram repetition rate: {pegasus_v2_bigram:.2f}%")
print(f"  Redundancy score: {pegasus_v2_redundancy:.2f} (higher = more redundant)")


REDUNDANCY CHECK:
ORIGINAL VERSIONS:
----------------------------------------------------------------------
TextRank:
  Average sentence similarity: 0.7355
  Bigram repetition rate: 52.83%
  Redundancy score: 60.62 (higher = more redundant)

LexRank:
  Average sentence similarity: 0.0720
  Bigram repetition rate: 11.39%
  Redundancy score: 7.72 (higher = more redundant)

DistilBART:
  Average sentence similarity: 0.0557
  Bigram repetition rate: 9.74%
  Redundancy score: 6.32 (higher = more redundant)

Pegasus:
  Average sentence similarity: 0.0644
  Bigram repetition rate: 5.97%
  Redundancy score: 5.43 (higher = more redundant)

V2 VERSIONS:
----------------------------------------------------------------------
TextRank V2:
  Average sentence similarity: 0.4366
  Bigram repetition rate: 37.93%
  Redundancy score: 39.78 (higher = more redundant)

LexRank V2:
  Average sentence similarity: 0.0573
  Bigram repetition rate: 10.70%
  Redundancy score: 6.90 (higher = more redundant)

Disti